# 🧰 03. 스킬 확장 & MCP (Skills & Model Context Protocol)

본 실습 노트북은 에이전트의 능력을 확장하는 **Skill (점진적 지침/워크플로우 모듈)** 과 **Model Context Protocol (MCP)** 의 2대 바인딩 아키텍처(**Static Process Binding** vs **Dynamic Context Binding**)를 단계별로 직접 실행하며 학습하는 자료입니다.

---

### 💡 핵심 아이디어

1. **Tool vs Skill 구분**
   - **Tool**: 모델 API의 `tools` 필드로 입력 스키마가 바인딩되는 **실행 인터페이스**(예: `file_read`, `bash_command`, 개별 API/MCP 도구).
   - **Skill**: 특정 과업을 일관되게 수행하는 **절차·지침·도메인 노하우를 담은 `SKILL.md` 중심 모듈**입니다. 지침·체크리스트 단독으로도 동작할 수 있고, 필요에 따라 `scripts/`·`references/`·`assets/`를 함께 패키징할 수도 있습니다.
2. **도구 과부하(Tool Overload) 극복과 Progressive Disclosure (점진적 노출)**
   - 수십~수백 개의 전문 도구를 모델 API의 `tools` 필드에 전량 바인딩하면 **스키마 토큰 비용·지연 시간이 증가하고 도구 선택 정확도가 떨어집니다**.
   - 따라서 최소한의 범용 기본 도구(`file_read`, `bash_command` 등)만 상시 바인딩하고, 시작 컨텍스트에는 각 스킬의 짧은 요약(`name`, `description`, `path`)만 두었다가 필요한 순간에만 `SKILL.md` 본문을 읽어 지침을 따르거나 동반 스크립트를 실행하는 **Progressive Disclosure(점진적 노출)** 패턴을 활용합니다.

---

### 🎓 학습 목차 (Curriculum Flow)

| 파트 | 주제 | 핵심 실습 내용 |
|:---:|:---|:---|
| **Step 0** | **환경 세팅 & 공통 헬퍼** | 루트 탐색, `.env` 로드, `nest_asyncio` 및 추적 헬퍼(`run_agent_trace`) 초기화 |
| **Part 1** | **Skills & Progressive Disclosure** | 1-1) `SkillPromptBuilder`의 YAML Frontmatter 스캔<br>1-2) 지침 + 카탈로그 결합 확인<br>1-3) 실전 테스트: 스킬만 보고 PDF 추출하기 |
| **Part 2** | **MCP 개념 & 2대 바인딩 아키텍처** | 2-1) Model Context Protocol 표준 통신 규격 (Client ↔ Server)<br>2-2) Static Process Binding vs Dynamic Context Binding 비교 |
| **Part 3** | **[패턴 1] Static Process Binding 실습** | 3-1) `langchain-mcp-adapters`(`MultiServerMCPClient`)로 MCP 도구 직접 바인딩<br>3-2) 직접 바인딩된 도구를 장착한 Static MCP 에이전트 실행 궤적 확인 |
| **Part 4** | **[패턴 2] Dynamic Context Binding (Progressive MCP)** | 4-1) 전용 MCP 도구 없이 `skills/mcp/SKILL.md` 기반 동적 탐색 & 실행<br>4-2) 도구 수 증가에 따른 토큰 오버헤드 정량 비교 (Static vs Dynamic) |
| **Part 5** | **[학습 정리] 도구 확장 의사결정 가이드** | Native Tool vs Skills (지침 단독/스크립트 동반) vs MCP 선택 기준 |

---

## 🛠️ Step 0. 환경 세팅 & 공통 추적 헬퍼

프로젝트 루트 전환, `.env` 로드, `nest_asyncio` 활성화 및 에이전트 실행 추적 헬퍼(`run_agent_trace`)를 초기화합니다.  
실습에 사용하는 샘플 PDF(`research_paper.pdf`)와 로컬 FastMCP 서버(`demo_mcp_server.py`)는 `artifacts/notebooks/data/`에 미리 준비되어 있습니다.

In [ ]:
import os, sys, json, asyncio, nest_asyncio
from dotenv import load_dotenv

nest_asyncio.apply()

def find_project_root():
    p = os.path.abspath(os.getcwd())
    while p != os.path.dirname(p):
        if os.path.exists(os.path.join(p, "app")) and (
            os.path.exists(os.path.join(p, ".env")) or os.path.exists(os.path.join(p, "configs"))
        ):
            return p
        p = os.path.dirname(p)
    return os.path.abspath(os.path.join(os.getcwd(), ".."))

project_root = find_project_root()
os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)
load_dotenv(os.path.join(project_root, ".env"), override=True)

from app.tools.common import file_read, bash_command
from app.utils import init_chat_model
from app.middleware.prompt import SkillPromptBuilder
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

async def run_agent_trace(agent, query: str, thread_id: str, label: str = "Agent"):
    print(f"\n{'='*60}\n🤖 [{label}] 실행 추적\n{'='*60}")
    final = ""
    async for chunk in agent.astream({"messages": [HumanMessage(content=query)]}, config={"configurable": {"thread_id": thread_id}}, stream_mode="values"):
        msgs = chunk.get("messages", [])
        if not msgs:
            continue
        msg = msgs[-1]
        if getattr(msg, "tool_calls", None):
            for tc in msg.tool_calls:
                print(f"\n👉 [Tool Call] {tc['name']} {json.dumps(tc['args'], ensure_ascii=False)}")
        elif isinstance(msg, ToolMessage):
            s = str(msg.content).strip()
            print(f"   ↳ [Tool Result] {s[:300]}{' ...' if len(s) > 300 else ''}")
        elif isinstance(msg, AIMessage) and msg.content:
            c = msg.content
            final = "\n".join(b["text"] for b in c if isinstance(b, dict) and b.get("type") == "text") if isinstance(c, list) else str(c)
    print(f"\n{'='*60}\n💬 [{label}] 최종 응답:\n{final}\n{'='*60}")

print(f"✅ 준비 완료 | Project Root: {project_root}")

## 📂 Part 1. Skills & Progressive Disclosure (점진적 스킬 노출)

### 1. Skill의 개념과 스펙트럼
**Skill**은 모델 API의 `tools` 필드에 등록하는 실행 인터페이스가 아니라, **특정 과업을 처리하는 절차·지침·도메인 노하우를 담은 `SKILL.md` 중심 모듈**입니다. 스킬의 형태는 크게 두 가지로 나뉩니다.
- **지침 단독형 스킬**: 별도 실행 스크립트 없이 `SKILL.md` 본문(예: 코드 리뷰 체크리스트, 보고서 작성 규칙, 분석 워크플로우)만으로 동작합니다. 필요할 때만 `file_read`로 지침을 읽어 컨텍스트에 반영합니다.
- **스크립트 동반형 스킬**: `SKILL.md`와 함께 `scripts/`(필요 시 `references/`, `assets/`) 디렉터리를 포함합니다. 에이전트는 `SKILL.md`에서 사용법과 인자 규약을 읽은 뒤 `bash_command`로 스크립트를 실행합니다.

### 2. Progressive Disclosure와 프롬프트 내 위치
- **Progressive Disclosure 원칙**:
  1. 에이전트에게는 범용 **Base Tools (`file_read`, `bash_command`)**만 상시 바인딩합니다.
  2. 각 스킬 폴더는 대문자 **`SKILL.md`** 파일의 상단 **YAML Frontmatter**(`name`, `description`)로 짧은 요약만 노출합니다.
  3. `SkillPromptBuilder`가 앞부분(약 2KB)만 고속 스캔하여 **초경량 카탈로그**를 만들고, 실제 실행 지침과 스크립트 사용법은 작업이 필요할 때 에이전트가 직접 `SKILL.md`를 열어 확인합니다.

```text
skills/
├── pdf_processing/
│   ├── SKILL.md          # YAML Frontmatter(name, description) + 절차/사용법
│   └── scripts/          # (선택) 보조 실행 스크립트 (pdf_extractor.py 등)
└── code_review/
    └── SKILL.md          # 스크립트 없이 지침·체크리스트 단독으로도 유효한 스킬!
```

> **💡 캐시 우선 `PromptAssembler`에서의 스킬 배치 (`2.Context_Engineering.ipynb` 연계)**
> 프로덕션 하네스(`PromptAssembler`)에서는 공용 `SystemMessage`의 KV-Cache를 전 사용자 간 100% 공유하기 위해, 스킬 가이드라인과 초기 스킬 카탈로그를 정적 시스템 프롬프트에 섞지 않고 **세션 시작 시 동결되는 `User Context`(`HumanMessage` `# skillCatalog`)**에 배치합니다. 이후 세션 도중 스킬이 추가·제거되더라도 `# skillCatalog`를 덮어쓰지 않고 대화 끝에 **`skill_delta` 메시지**만 append하여 앞단 캐시를 유지합니다.

In [ ]:
# Part 1-1. SkillPromptBuilder의 YAML Frontmatter 스캔 및 카탈로그 자동 생성 실습

# 1. SkillPromptBuilder 인스턴스 생성 (skills/ 디렉터리 스캔)
guidelines_file = os.path.join(project_root, "app/prompts/SKILL.md")
skill_builder = SkillPromptBuilder(
    skills_dirs=[os.path.join(project_root, "skills")],
    guidelines_path=guidelines_file if os.path.exists(guidelines_file) else None,
)

# 2. YAML Frontmatter 파싱 결과(구조화 메타데이터) 확인
scanned_skills = skill_builder.list_skills()
print(f"🔎 YAML Frontmatter 스캔 완료: 총 {len(scanned_skills)}개 스킬 발견\n")
for s in scanned_skills:
    print(f"- name: {s['name']}")
    print(f"  path: {s['path']}")
    print(f"  desc: {s['description']}\n")

In [ ]:
# 3. 스캔된 Skills 카탈로그 마크다운 확인
catalog_text = skill_builder.build_catalog()
print("📦 [스캔된 Skills 카탈로그]")
print(catalog_text)
print("\n" + "-" * 50 + "\n")

### 1-2. 스킬 지침(Guidelines) + 카탈로그 결합 확인

`skill_builder.assemble()`은 `app/prompts/SKILL.md` 지침과 스캔된 카탈로그를 하나의 문자열로 합쳐 주므로 단독 테스트에 편리합니다.  
(참고: 실제 프로덕션 하네스에서는 `PromptAssembler(skill_guidelines=skill_builder.load_guidelines, skill_provider=skill_builder.list_skills)` 형태로 연결하여 `User Context`의 `# skillCatalog` 블록에 조립합니다.)

In [ ]:
# 4. 가이드라인 + 카탈로그 결합 문자열 전체 확인
full_skill_prompt = skill_builder.assemble()
print(f"📋 [조립된 스킬 지침 + 카탈로그 (전체 {len(full_skill_prompt)}자)]")
print(full_skill_prompt)

### 1-3. 실전 테스트: 스킬(`SKILL.md`)만 보고 스스로 PDF 텍스트 추출하기

에이전트에게 **오직 `[file_read, bash_command]` 2개 기본 도구만** 부여하고, `artifacts/notebooks/data/research_paper.pdf`를 처리하기 위해 `skills/pdf_processing/SKILL.md`를 스스로 찾아 읽어 추출하도록 지시합니다.

In [ ]:
# Part 1-3. Base Tools 기반 Progressive Skill 실행 실습
llm = init_chat_model(model="gemini-3.8-flash", temperature=0.0)
demo_pdf_path = "artifacts/notebooks/data/research_paper.pdf"

# (단일 턴 데모이므로 system_prompt에 간단히 전달하며, 멀티턴 프로덕션에서는 PromptAssembler가 User Context에 배치합니다.)
system_instruction = f"""You are an expert AI agent with progressive skill execution capabilities.
You ONLY have 'file_read' and 'bash_command' base tools.

{full_skill_prompt}

When asked to process a file with a relevant skill:
1. Check Available Skills and read its SKILL.md documentation via 'file_read'.
2. Run the documented python script via 'bash_command'.
3. Summarize the extracted result clearly in Korean.
"""

progressive_agent = create_agent(
    model=llm,
    tools=[file_read, bash_command],
    system_prompt=system_instruction,
    checkpointer=MemorySaver(),
)

user_query = f"'{demo_pdf_path}' 논문 파일의 제목, 저자, 페이지 수 및 주요 요약 내용을 추출해서 한글로 정리해줘."
asyncio.run(run_agent_trace(progressive_agent, user_query, "skill_demo_session_1", "Progressive Skill Agent"))

## 🔌 Part 2. Model Context Protocol (MCP) 개요 & 2대 바인딩 모델

**Model Context Protocol (MCP)** 은 AI 에이전트(Client)와 외부 도구·데이터 소스(Server)를 **JSON-RPC 표준 규약**(Stdio / SSE / Streamable HTTP)으로 연결하는 개방형 프로토콜입니다.  
MCP 서버를 에이전트 하네스에 통합할 때는 목적과 도구 수에 따라 크게 두 가지 바인딩 패턴을 선택합니다.

### 1. Static Process Binding vs Dynamic Context Binding 비교

| 비교 항목 | ⚙️ 패턴 1: Static Process Binding (정적 바인딩) | ⚡ 패턴 2: Dynamic Context Binding (동적 바인딩) |
|:---|:---|:---|
| **핵심 메커니즘** | `langchain-mcp-adapters` 등을 통해 MCP 서버의 도구를 에이전트 **`tools` 스키마에 직접 바인딩** | MCP 전용 도구를 바인딩하지 않고, **기본 도구(`file_read`, `bash_command`) + `skills/mcp/SKILL.md` 컨텍스트**로 연결 |
| **초기 컨텍스트 비용** | API `tools` 필드에 연결된 모든 MCP 도구 스키마가 매 턴 포함됨 (**도구 수 N에 비례**) | 서버/도구가 늘어나도 기본 도구 2종 + 스킬 카탈로그 1줄만 고정 유지 (**상수 시간 O(1)**) |
| **추론 레이턴시** | 모델이 `calculate_roi(...)` 같은 도구를 **단일 턴으로 즉시 호출** | `SKILL.md` 열람 → `list_tools.py` → `execute_tool.py` 순으로 **추가 추론 턴 발생** |
| **세션 상태 유지** | `client.session(...)` 기반 **지속 세션(Stateful)** 유지 용이 | CLI 스크립트 1회 호출마다 연결을 맺고 종료하는 **Stateless** 구조에 적합 |
| **권장 사용처** | 자주 쓰이고 상태 유지가 중요한 **소수의 핵심 MCP 서버(도구 ~10개 내외)** | 다수의 외부 MCP 서버·**수십~수백 개 도구 생태계**를 가볍게 온디맨드 연동할 때 |

## ⚙️ Part 3. [패턴 1] Static Process Binding 실습 (`langchain-mcp-adapters`)

미리 준비된 로컬 FastMCP 서버(`artifacts/notebooks/data/demo_mcp_server.py`)에 `langchain-mcp-adapters`(`MultiServerMCPClient` + `load_mcp_tools`)로 연결하여 `calculate_roi`, `query_database` 도구를 **에이전트의 `tools`에 직접 바인딩**합니다.

In [ ]:
# Part 3-1. MultiServerMCPClient로 MCP 서버 도구 직접 발견 및 LangChain Tool 스키마 확인
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.tools import load_mcp_tools

demo_mcp_server_path = "artifacts/notebooks/data/demo_mcp_server.py"
demo_server_cmd = f"{sys.executable} {demo_mcp_server_path}"

mcp_client = MultiServerMCPClient({
    "demo_data_engine": {
        "transport": "stdio",
        "command": sys.executable,
        "args": [os.path.join(project_root, demo_mcp_server_path)],
    }
})


async def inspect_static_mcp_tools():
    async with mcp_client.session("demo_data_engine") as session:
        tools = await load_mcp_tools(session)
        print(f"✅ Static MCP 바인딩으로 발견된 직접 도구 ({len(tools)}개):")
        for t in tools:
            print(f"  - {t.name}: {t.description}")
            print(f"    schema keys: {list((t.args_schema or {}).get('properties', {}).keys())}")
        return [t.name for t in tools]


bound_tool_names = asyncio.run(inspect_static_mcp_tools())

In [ ]:
# Part 3-2. Static Process Binding 에이전트 실행 (지속 MCP 세션 내 직접 도구 호출)

async def run_static_mcp_mission():
    static_mcp_prompt = (
        "You are an analytical assistant directly connected to the Data & Math Engine MCP tools. "
        "Use the bound database and ROI tools accurately to fulfill the user's request in Korean."
    )
    user_query_mcp = (
        "sales 테이블의 매출 데이터를 조회한 뒤, "
        "Q4 분기 매출(Revenue)과 비용(Cost=$1,200,000)을 기준으로 ROI를 계산해서 최종 보고해줘."
    )

    # 지속 세션(async with mcp_client.session) 안에서 도구를 로드해 에이전트를 실행하면
    # 매 Tool Call마다 서버 프로세스를 재시작하지 않고 동일 MCP 세션을 재사용할 수 있습니다.
    async with mcp_client.session("demo_data_engine") as session:
        static_mcp_tools = await load_mcp_tools(session)
        static_mcp_agent = create_agent(
            model=llm,
            tools=static_mcp_tools,
            system_prompt=static_mcp_prompt,
            checkpointer=MemorySaver(),
        )
        await run_agent_trace(static_mcp_agent, user_query_mcp, "static_mcp_session", "Static MCP Binding Agent")


asyncio.run(run_static_mcp_mission())

## ⚡ Part 4. [패턴 2] Dynamic Context Binding (Progressive MCP) 실습

이번에는 에이전트에게 **MCP 전용 도구를 일체 바인딩하지 않고**, 오직 **기본 도구(`file_read`, `bash_command`)**, `SkillPromptBuilder` 카탈로그, 그리고 연결 가능한 **MCP 서버 목록(이름·대상 주소/실행 명령)**만 컨텍스트로 제공합니다.

핵심 포인트는 프롬프트에 `list_tools.py` / `execute_tool.py` CLI 문법을 미리 적어두지 않는 것입니다.  
에이전트가 스스로 **`skills/mcp/SKILL.md`를 `file_read`로 열어 CLI 사용법을 학습**한 뒤 `bash_command`로 MCP 서버 도구를 탐색·실행하는 진짜 Progressive Disclosure 흐름을 확인합니다.

In [ ]:
# Part 4-1. Dynamic Context Binding (Progressive MCP) 실습

# 1. 연결 가능한 MCP 서버 카탈로그 정의 (구체적 스크립트 CLI 문법은 포함하지 않음!)
mcp_server_catalog = f"""# Available MCP Servers
- **Data & Math Engine**
  - Target (`--url`): `{demo_server_cmd}`
  - Description: Local Stdio MCP server providing corporate database queries and financial ROI calculation tools."""

# 2. Progressive MCP 지침 구성 (스킬 카탈로그 + MCP 서버 카탈로그만 제공)
dynamic_mcp_prompt = f"""You are a progressive skill-disclosure agent.
You only have 'file_read' and 'bash_command' base tools, with no dedicated MCP tools bound directly.

{full_skill_prompt}

{mcp_server_catalog}

Guidelines:
1. To interact with an MCP server above, look up the `mcp-client` skill in Available Skills and read `skills/mcp/SKILL.md` via `file_read` first.
2. Follow the documented scripts and arguments in `skills/mcp/SKILL.md` using `bash_command` to list tools first and then execute the required tool(s).
3. Present the final report clearly in Korean.
"""

# 3. 오직 Base Tools만 장착한 에이전트 생성 및 동일 미션 실행
dynamic_mcp_agent = create_agent(
    model=llm,
    tools=[file_read, bash_command],
    system_prompt=dynamic_mcp_prompt,
    checkpointer=MemorySaver(),
)

user_query_dynamic = (
    "Data & Math Engine MCP 서버에서 sales 테이블의 데이터를 조회하고, "
    "Q4 분기 매출과 비용(Cost=$1,200,000)에 대한 ROI를 계산해서 한글로 보고해줘."
)
asyncio.run(run_agent_trace(dynamic_mcp_agent, user_query_dynamic, "dynamic_mcp_session", "Dynamic Context MCP Agent"))

### 4-2. 도구 수 증가에 따른 토큰 오버헤드 정량 비교 (Static vs Dynamic)

실제 LLM API에 전달되는 OpenAI/LangChain `tools` JSON 스키마 크기(`convert_to_openai_tool`)를 기준으로, **MCP 도구가 N = 2, 10, 50, 100개로 늘어날 때** 두 아키텍처의 초기 컨텍스트 비용이 어떻게 달라지는지 오프라인으로 측정합니다.

In [ ]:
# Part 4-2. MCP 도구 수(N) 증가에 따른 초기 컨텍스트/스키마 크기 비교 (오프라인 측정)
from langchain_core.utils.function_calling import convert_to_openai_tool


async def measure_schema_bytes():
    async with mcp_client.session("demo_data_engine") as session:
        mcp_tools = await load_mcp_tools(session)
        return [len(json.dumps(convert_to_openai_tool(t), ensure_ascii=False)) for t in mcp_tools]


mcp_tool_schema_chars = asyncio.run(measure_schema_bytes())
avg_mcp_tool_chars = sum(mcp_tool_schema_chars) / len(mcp_tool_schema_chars)

base_tools_schema_chars = sum(
    len(json.dumps(convert_to_openai_tool(t), ensure_ascii=False))
    for t in [file_read, bash_command]
)
server_catalog_text = globals().get("mcp_server_catalog") or (
    f"# Available MCP Servers\n- **Data & Math Engine**\n  - Target (`--url`): `{demo_server_cmd}`\n"
    "  - Description: Local Stdio MCP server providing corporate database queries and financial ROI calculation tools."
)
skill_catalog_chars = len(full_skill_prompt)
server_catalog_chars = len(server_catalog_text)
dynamic_baseline_chars = base_tools_schema_chars + skill_catalog_chars + server_catalog_chars

print("📏 실측 기준치:")
print(f"  - MCP 도구 1개당 평균 JSON 스키마 크기 : {avg_mcp_tool_chars:,.0f}자 (약 {avg_mcp_tool_chars / 4:,.0f} 토큰)")
print(f"  - Dynamic 기본 고정 오버헤드          : {dynamic_baseline_chars:,}자 (Base Tools {base_tools_schema_chars:,} + 스킬/서버 카탈로그 {skill_catalog_chars + server_catalog_chars:,})\n")

print(f"{'도구 수(N)':>10} | {'Static 직접 바인딩(자 / ~토큰)':>30} | {'Dynamic 고정 컨텍스트(자 / ~토큰)':>32} | {'초기 절감률':>10}")
print("-" * 92)
for n in (2, 10, 50, 100):
    static_chars = int(round(avg_mcp_tool_chars * n))
    static_tok = static_chars // 4
    dyn_tok = dynamic_baseline_chars // 4
    saving = (1 - dynamic_baseline_chars / static_chars) * 100 if static_chars else 0.0
    print(f"{n:>9}개 | {static_chars:>12,}자 (~{static_tok:>6,} tok) | {dynamic_baseline_chars:>14,}자 (~{dyn_tok:>6,} tok) | {saving:>9.1f}%")

print("\n💡 해석:")
print("  - 도구가 2개뿐인 소규모 환경에서는 Static 직접 바인딩의 스키마 비용이 오히려 가볍고 LLM 호출 횟수도 적습니다.")
print("  - 하지만 도구가 10개 → 50개 → 100개로 늘어나면 Static 바인딩은 O(N)으로 팽창하는 반면, Dynamic 바인딩은 초기 컨텍스트를 O(1)로 고정합니다.")

## 📝 Part 5. [학습 정리] 프로덕션 도구 확장 의사결정 가이드

실무 에이전트 하네스에서 기능을 확장할 때는 아래 3가지 계층을 목적에 맞게 조합합니다.

```text
┌─────────────────────────────────────────────────────────────────────────────┐
│ 1. 범용 핵심 도구 (파일·셸·검색)  ➔ 네이티브 LangChain Tool 직접 바인딩     │
│    • file_read, file_writer, bash_command, glob_search, grep_search 등       │
│    • 거의 매 작업마다 호출되므로 API tools 필드에 상시 바인딩               │
├─────────────────────────────────────────────────────────────────────────────┤
│ 2. 절차·도메인 노하우 / CLI 패키지 ➔ Skills (SKILL.md + 선택적 scripts/)    │
│    • [지침 단독] 코드 리뷰 가이드, 리포트 템플릿, 작업 체크리스트           │
│    • [스크립트 동반] PDF 추출, 데이터 프로파일링, 차트 생성                 │
│    • Frontmatter 카탈로그로만 노출하고 필요할 때만 SKILL.md 본문 로드       │
├─────────────────────────────────────────────────────────────────────────────┤
│ 3. 외부 표준 시스템 연동         ➔ MCP (Static vs Dynamic 선택)             │
│    • [Static Process Binding] 빈번하고 상태 유지가 필요한 소수 핵심 도구    │
│    • [Dynamic Context Binding] 수십~수백 개 외부 MCP 서버를 가볍게 온디맨드 │
└─────────────────────────────────────────────────────────────────────────────┘
```